# SHORT_SQUEEZE on ETH (2026-09)

**Question (roadmap §3 research item 1, second assets):** does the shipped SHORT_SQUEEZE rule — the sweep of the
prior 24-bar low with perp-CVD selling and spot-vs-perp divergence on a short-macro day, London/NY, long only —
earn on ETH at the measured 10 bp round trip, and does the twin-table builder reproduce the BTC run first?

This notebook checks the frozen run: it reloads `results/report.json` and `results/trades_*.csv`, recomputes the
ETH statistics from the trades and asserts they match the report. Design: [README.md](README.md) (frozen,
`results/freeze_F0.json`). Verdict and reading: [findings.md](findings.md).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); sys.path.insert(0, str(HERE))
import ss_eth_lib as L
R = HERE / "results"
report = json.loads((R / "report.json").read_text())
f0 = json.loads((R / "freeze_F0.json").read_text())
print("frozen", f0["created_utc"], "| outcome", report["created_utc"])
print("P0 (engine == port, anchor):", {k: v for k, v in report["P0"].items() if k != "anchor_expected"})
print("P1 (builder fidelity on BTC):", {k: v for k, v in report["P1"].items() if k not in ("only_a", "only_b")})
print("DECISION:", report["decision"])

frozen 2026-09-18T23:49:57+00:00 | outcome 2026-09-18T23:49:59+00:00
P0 (engine == port, anchor): {'engine_equals_port': True, 'anchor_got': {'n': 70, 'win': 0.45714285714285713, 'mean_r': 0.3916288386832697, 'pf': 1.64431729954119}, 'anchor_pass': False, 'pass': False}
P1 (builder fidelity on BTC): {'common_span': ['2022-01-30 06:00:00+00:00', '2026-09-13 20:45:00+00:00'], 'a': 71, 'b': 72, 'shared': 71, 'jaccard': 0.9861111111111112, 'shared_trades': 71, 'max_abs_pnl_diff_R': 0.0, 'pnl_agree': True, 'pass': True}
DECISION: {'a_n': True, 'b_net_and_ci': False, 'c_both_halves': False, 'd_dsr': False, 'e_fidelity': True, 'failing': ['b_net_and_ci', 'c_both_halves', 'd_dsr'], 'verdict': 'KILL for ETH'}


## 1. The three runs side by side

`BTC_prod`: the engine on prod's tables (the anchor). `BTC_panel`: the same engine on tables built from the on-disk
panels — the fidelity run. `ETH_panel`: the same builder and engine on ETH. Gross is 0 bp per leg; net charges the
10 bp round trip against each trade's own risk distance.

In [2]:
rows = {}
for label in ("BTC_prod", "BTC_panel", "ETH"):
    rec = report[label]
    for form in ("gross", "net"):
        d = rec[form]
        if not d.get("n"):
            rows[f"{label} {form}"] = {"n": 0}; continue
        rows[f"{label} {form}"] = {"n": d["n"], "mean R": round(d["mean_R"], 3), "win": round(d["win_rate"], 3),
                                   "PF": None if d["pf"] is None else round(d["pf"], 2),
                                   "halves": f"{d['first_half_mean_R']:+.2f} / {d['second_half_mean_R']:+.2f}",
                                   "annual R": round(d["annual_R"], 1), "max DD R": round(d["max_dd_R"], 1),
                                   "MAR": None if d["mar"] is None else round(d["mar"], 2),
                                   "DSR": None if d.get("dsr") is None else round(d["dsr"], 3),
                                   "median risk %": round(d["median_risk_pct"], 2), "exits": d["exit_mix"],
                                   "span": rec["frame_span"][0][:10] + " → " + rec["frame_span"][1][:10]}
    rows[f"{label} net"]["CI90 mean"] = [round(x, 3) for x in rec["boot_net"]["ci90"]] if rec["boot_net"]["ci90"][0] is not None else None
    rows[f"{label} net"]["cost share of gross"] = None if rec["cost_share_of_gross"] is None else round(rec["cost_share_of_gross"], 2)
pd.DataFrame(rows).T

,n,mean R,win,PF,halves,annual R,max DD R,MAR,DSR,median risk %,exits,span,CI90 mean,cost share of gross
BTC_prod gross,71,0.498,0.451,1.91,+0.41 / +0.59,8.8,7.0,1.25,0.994,0.34,"{'stop': 39, 'target': 21, 'time': 11}",2022-01-30 → 2026-09-18,NaN,NaN
BTC_prod net,71,0.179,0.451,1.25,+0.14 / +0.21,3.2,8.4,0.37,0.809,0.34,"{'stop': 39, 'target': 21, 'time': 11}",2022-01-30 → 2026-09-18,"[-0.14, 0.496]",0.64
BTC_panel gross,81,0.412,0.42,1.71,+0.31 / +0.52,6.8,7.0,0.97,0.987,0.36,"{'stop': 47, 'target': 23, 'time': 11}",2020-09-01 → 2026-09-13,NaN,NaN
BTC_panel net,81,0.107,0.42,1.14,+0.07 / +0.14,1.8,8.4,0.21,0.711,0.36,"{'stop': 47, 'target': 23, 'time': 11}",2020-09-01 → 2026-09-13,"[-0.215, 0.442]",0.74
ETH gross,70,0.344,0.386,1.56,+0.56 / +0.13,5.7,7.0,0.8,0.956,0.54,"{'stop': 43, 'target': 18, 'time': 9}",2021-12-01 → 2026-09-13,NaN,NaN
ETH net,70,0.138,0.386,1.19,+0.37 / -0.09,2.3,10.6,0.22,0.747,0.54,"{'stop': 43, 'target': 18, 'time': 9}",2021-12-01 → 2026-09-13,"[-0.207, 0.483]",0.6


In [3]:
eth = pd.read_csv(R / "trades_ETH_panel.csv")
mine = L.describe(L.with_net(eth[["trigger_ts", "entry", "stop", "target", "exit_price", "exit_reason", "pnl_R", "risk_pct"]]), "net_R")
saved = report["ETH"]["net"]
if saved.get("n"):
    assert mine["n"] == saved["n"] and abs(mine["mean_R"] - saved["mean_R"]) < 1e-9 and abs(mine["max_dd_R"] - saved["max_dd_R"]) < 1e-9
    print("recomputed the ETH net statistics from trades_ETH_panel.csv and matched the report")
print("ETH per year (net):", json.dumps(saved.get("per_year", {}), indent=1))

recomputed the ETH net statistics from trades_ETH_panel.csv and matched the report
ETH per year (net): {
 "2022": {
  "n": 38,
  "mean_R": 0.35083658739807483,
  "sum_R": 13.331790321126844
 },
 "2023": {
  "n": 12,
  "mean_R": -0.007235927822251483,
  "sum_R": -0.0868311338670178
 },
 "2025": {
  "n": 11,
  "mean_R": -0.2269033348952202,
  "sum_R": -2.495936683847422
 },
 "2026": {
  "n": 9,
  "mean_R": -0.1177512000644801,
  "sum_R": -1.059760800580321
 }
}


## 2. The ETH equity path against BTC's

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
for label, color in (("BTC_prod", "tab:gray"), ("BTC_panel", "tab:blue"), ("ETH_panel", "tab:orange")):
    t = pd.read_csv(R / f"trades_{label}.csv")
    if len(t):
        t["trigger_ts"] = pd.to_datetime(t["trigger_ts"], utc=True)
        ax.plot(t["trigger_ts"], t["net_R"].cumsum(), lw=1.2, label=f"{label} ({len(t)} trades)", color=color)
ax.set_ylabel("cumulative net R (10 bp round trip)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("SHORT_SQUEEZE: the same engine on BTC (prod tables, panel tables) and on ETH")
plt.show()

C:\Users\TJ5\AppData\Local\Temp\ipykernel_24560\2409090644.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
print(json.dumps(report["decision"], indent=1))

{
 "a_n": true,
 "b_net_and_ci": false,
 "c_both_halves": false,
 "d_dsr": false,
 "e_fidelity": true,
 "failing": [
  "b_net_and_ci",
  "c_both_halves",
  "d_dsr"
 ],
 "verdict": "KILL for ETH"
}
